# Data Understanding: fuentes de maiz (Argentina)

Objetivo: cargar y entender cada CSV original de `AR/maize` de forma independiente. En esta etapa no se unen fuentes, no se imputan faltantes y no se alteran unidades ni fechas. Los archivos grandes se recorren por bloques para obtener un perfil completo sin mantenerlos todos en memoria.

## Alcance y preguntas iniciales

El proyecto configura nueve fuentes de maiz. Hay tambien archivos de trigo en el repositorio, pero quedan fuera de esta primera exploracion porque la configuracion activa apunta a `AR/maize`. Las preguntas iniciales son: que representa una fila en cada archivo, que columnas y tipos trae, que cobertura temporal tiene, y donde hay valores faltantes o rangos que requieran validacion.

In [ ]:
from pathlib import Path
import sys

import pandas as pd
from IPython.display import display

ROOT_DIR = Path.cwd()
if not (ROOT_DIR / "src").is_dir():
    ROOT_DIR = ROOT_DIR.parent
if str(ROOT_DIR / "src") not in sys.path:
    sys.path.insert(0, str(ROOT_DIR / "src"))

from maize_forecasting.config import INPUT_FILES, MAIZE_DATA_DIR
from maize_forecasting.data_understanding import load_dataset, profile_dataset

print(f"Proyecto: {ROOT_DIR}")
print(f"Directorio de datos: {MAIZE_DATA_DIR}")

## Inventario de archivos

El perfilado lee cada CSV por separado. `load_dataset` tambien permite cargar un archivo individual completo cuando se necesite inspeccionarlo en detalle; para esta primera pasada usamos una muestra pequeña y un perfil en streaming.

In [ ]:
inventory = pd.DataFrame(
    [
        {
            "dataset": name,
            "file": filename,
            "exists": (MAIZE_DATA_DIR / filename).is_file(),
            "size_mib": (MAIZE_DATA_DIR / filename).stat().st_size / 1024**2,
        }
        for name, filename in INPUT_FILES.items()
    ]
)
display(inventory)

## Carga de muestras sin combinar

Se leen las primeras cinco filas de cada fuente. Cada tabla se conserva bajo su propio nombre para hacer visibles sus diferencias de estructura y granularidad.

In [ ]:
samples = {name: load_dataset(name, nrows=5) for name in INPUT_FILES}
for name, sample in samples.items():
    print(f"{name}: {sample.shape[0]} filas de muestra, {sample.shape[1]} columnas")
    display(sample)

## Perfilado completo por CSV

El siguiente recorrido cuenta todas las filas y faltantes y calcula rangos numericos globales, pero solo mantiene un bloque de cada archivo en memoria. No calcula relaciones entre tablas ni presupone que sus claves coinciden.

In [ ]:
profiles = [profile_dataset(name, chunksize=100_000) for name in INPUT_FILES]
overview = pd.DataFrame(
    [
        {
            "dataset": profile["dataset"],
            "rows": profile["rows"],
            "columns": len(profile["columns"]),
            "distinct_adm_id": profile["distinct_counts"].get("adm_id"),
            "missing_cells": sum(profile["missing_values"].values()),
            "file_size_mib": round(profile["file_size_mib"], 2),
        }
        for profile in profiles
    ]
)
display(overview)

In [ ]:
schema = pd.DataFrame(
    [
        {"dataset": profile["dataset"], "column": column, "dtype": dtype}
        for profile in profiles
        for column, dtype in profile["dtypes"].items()
    ]
)
display(schema)

In [ ]:
missingness = pd.DataFrame(
    [
        {"dataset": profile["dataset"], "column": column, "missing": count, "missing_pct": profile["missing_percent"][column]}
        for profile in profiles
        for column, count in profile["missing_values"].items()
        if count > 0
    ]
)
display(missingness if not missingness.empty else "No se detectaron faltantes en los CSV perfilados.")

## Rangos numericos y primeras impresiones

Los rangos son descriptivos, no una validacion de unidades o plausibilidad. En particular, las fechas aparecen como enteros `YYYYMMDD` en algunas fuentes; se mantienen asi en la carga inicial.

Primeras impresiones del perfil completo:

- `yield` tiene 17,115 filas y cubre campanas con `harvest_year` entre 1970 y 2023.
- Meteorologia tiene 3,855,600 filas (2001-2023); humedad de suelo, 3,498,204 (2003-2023); NDVI y FPAR tambien empiezan en 2001 y llegan a 2023, con menos observaciones. Estas diferencias sugieren frecuencias/coberturas que deben analizarse por fuente.
- Las fuentes `soil`, `crop_calendar`, `location` y `crop_mask` tienen entre 445 y 526 filas, frente a cientos de miles o millones en las series fechadas. Son estructuras de menor granularidad, pero su clave y cobertura deben validarse individualmente.
- El escaneo no detecto celdas NA en ninguno de los nueve CSV. Esto no descarta codigos sentinela ni valores invalidos.
- Los rangos de variables como `rad` y `rsm` deben contrastarse con documentacion de unidades antes de interpretarlos.

La cobertura del objetivo empieza antes que la de las fuentes ambientales. Por eso, una futura tabla de entrenamiento probablemente no tendra el mismo periodo que `yield`; no hay que resolverlo uniendo ni descartando filas en esta etapa. No se realizan joins ni se asume que los conteos de `adm_id` validan por si solos la equivalencia espacial.

In [ ]:
for profile in profiles:
    print(f"{profile['dataset']} | {profile['rows']:,} filas | {profile['file_size_mib']:.2f} MiB")
    ranges = pd.DataFrame.from_dict(profile["numeric_ranges"], orient="index", columns=["min", "max"])
    display(ranges)

## Recibo de ejecucion

Se verifican cantidad de fuentes perfiladas, filas leidas y faltantes reportados. Las fuentes siguen separadas; no se ha construido ninguna tabla analitica integrada.

In [ ]:
total_rows_scanned = sum(profile["rows"] for profile in profiles)
total_missing = sum(sum(profile["missing_values"].values()) for profile in profiles)
print(f"Fuentes perfiladas: {len(profiles)}")
print(f"Filas leidas en total (sin combinar): {total_rows_scanned:,}")
print(f"Celdas faltantes detectadas: {total_missing:,}")